# Data Ingestion, Chunking, and Question Generation

This notebook processes raw documents and generates training data in two stages:

### Stage 1: Document Chunking
Load raw PDF and CSV files from the inbox, split them into manageable chunks using deterministic chunking, then archive the processed files and store chunks in a Delta table.

### Stage 2: Question Generation  
Generate synthetic training questions for each chunk using an LLM, optimized with batch processing and rate limiting.

---

**Inputs:** Raw PDF and CSV files in Unity Catalog Volumes (`bronze/inbox/CSV` and `bronze/inbox/PDF`)

**Outputs:** 
* `chunks` Delta table - Processed document chunks with metadata
* `questions` Delta table - LLM-generated questions for each chunk
* Archived source files and processing manifests

## Setup: Dependencies & Configuration

Installing required libraries for document processing, text chunking, and LLM integration.

In [0]:
%pip install langchain langchain-community pypdf unstructured tenacity openai
dbutils.library.restartPython()

In [0]:
# NVIDIA API credentials for LLM question generation
# Get your API key from https://build.nvidia.com
api = "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"  # <-- NVIDIA API key goes here
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_MODEL = "openai/gpt-oss-120b"

In [0]:
# Import all required libraries for document processing, chunking, and LLM integration
import os
import json
import gzip
import uuid
import hashlib
import logging
import re
import threading
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import pyspark.sql.functions as F

import openai
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

spark = SparkSession.builder.getOrCreate()

In [0]:
# Configure Unity Catalog paths and chunking parameters
# All data flows through the bronze layer with organized inbox/archive structure
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "code_v1"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

PATHS = {
    "inbox_csv":      f"{BASE}/bronze/inbox/CSV",
    "inbox_pdf":      f"{BASE}/bronze/inbox/PDF",
    "archive_csv":    f"{BASE}/bronze/inbox/outbox/archive/CSV",
    "archive_pdf":    f"{BASE}/bronze/inbox/outbox/archive/PDF",
    "chunks":         f"{BASE}/bronze/chunks",
    "questions":      f"{BASE}/bronze/questions",
    "gd_pairs":       f"{BASE}/bronze/gd_pairs",
    "answers":        f"{BASE}/bronze/answers",
    "datasets":       f"{BASE}/silver/pub/datasets",
    "manifests":      f"{BASE}/bronze/_manifests",
    "logs":           f"{BASE}/bronze/_logs",
}

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
QUESTIONS_PER_CHUNK = 3
NAMESPACE = uuid.NAMESPACE_OID

# API rate limiting and batching
API_RATE_LIMIT = 30          # calls per minute (max parallel calls per wave)
BATCH_SIZE = 20              # chunks per API call (10 chunks = 30 questions per call)
FALLBACK_BATCH_SIZE = 5      # fallback size if batches fail
MIN_WAVE_INTERVAL = 60.0     # seconds between waves (1 minute for rate limit)
MAX_MODEL_TOKENS = 90000    # Model's max context window

## Core Utility Functions

Helper functions for logging, directory management, and manifest tracking to ensure idempotent processing.

In [0]:
# Set up logging and helper functions for directory management and manifest tracking
# Manifests track processed files to prevent duplicate processing
def setup_logging():
    logger = logging.getLogger("chunks_questions")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        ch.setFormatter(formatter)
        logger.addHandler(ch)
    return logger

logger = setup_logging()

def ensure_dir(path: str):
    try:
        dbutils.fs.mkdirs(path)
    except Exception as e:
        logger.error(f"Failed to create directory {path}: {e}")
        raise

# Initialize base directories
for key, path in PATHS.items():
    ensure_dir(path)

def load_manifest(name: str) -> dict:
    manifest_path = f"{PATHS['manifests']}/{name}.json"
    if os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            return json.load(f)
    return {}

def save_manifest(name: str, entries: dict):
    manifest_path = f"{PATHS['manifests']}/{name}.json"
    with open(manifest_path, "w") as f:
        json.dump(entries, f, indent=2)

def hash_file(file_path: str) -> str:
    h = hashlib.sha256()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

## Stage 1: Document Chunking Pipeline

Process raw PDF and CSV files: discover new files, load and chunk documents, then archive processed files.

In [0]:
# Scan inbox folders for new files and check against the manifest to avoid reprocessing
file_manifest = load_manifest("processed_files")
new_files = []

for file_type, folder in [("PDF", PATHS["inbox_pdf"]), ("CSV", PATHS["inbox_csv"])]:
    if not os.path.exists(folder):
        continue
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        if os.path.isfile(file_path):
            file_hash = hash_file(file_path)
            if file_hash not in file_manifest:
                new_files.append((file_path, file_type, file_hash, filename))

logger.info(f"Discovered {len(new_files)} new files to process.")

In [0]:
# Load PDF and CSV files using LangChain loaders, attaching metadata to each document
def load_pdf(path: str, doc_id: str) -> list:
    loader = PyPDFLoader(path)
    docs = loader.load()
    for d in docs:
        d.metadata = {"source_path": path, "source_type": "PDF", "doc_id": doc_id}
    return docs

def load_csv(path: str, doc_id: str) -> list:
    loader = CSVLoader(path)
    docs = loader.load()
    for d in docs:
        d.metadata = {"source_path": path, "source_type": "CSV", "doc_id": doc_id}
    return docs

In [0]:
# Split documents into overlapping chunks with deterministic IDs for reproducibility
def chunk_documents(docs: list) -> list:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    split_docs = splitter.split_documents(docs)
    
    chunk_rows = []
    now_utc = datetime.now(timezone.utc)
    for i, doc in enumerate(split_docs):
        doc_id = doc.metadata["doc_id"]
        chunk_id = str(uuid.uuid5(NAMESPACE, f"{doc_id}:{i}"))
        chunk_text = doc.page_content
        
        chunk_rows.append({
            "chunk_id": chunk_id,
            "doc_id": doc_id,
            "source_path": doc.metadata["source_path"],
            "source_type": doc.metadata["source_type"],
            "chunk_index": i,
            "chunk_text": chunk_text,
            "char_count": len(chunk_text),
            "created_at_utc": now_utc
        })
    return chunk_rows

In [0]:
# Process all discovered files: load, chunk, and write to Delta table
chunks_schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("doc_id", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("chunk_index", IntegerType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("char_count", IntegerType(), False),
    StructField("created_at_utc", TimestampType(), False)
])

all_chunk_rows = []
successful_files = []
failed_files = []

for file_path, file_type, file_hash, filename in new_files:
    try:
        if file_type == "PDF":
            docs = load_pdf(file_path, file_hash)
        else:
            docs = load_csv(file_path, file_hash)
            
        file_chunk_rows = chunk_documents(docs)
        all_chunk_rows.extend(file_chunk_rows)
        successful_files.append((file_path, file_type, file_hash, filename))
        logger.info(f"Successfully chunked {filename} into {len(file_chunk_rows)} chunks.")
    except Exception as e:
        logger.error(f"Failed to process file {filename}: {e}")
        failed_files.append(filename)

if all_chunk_rows:
    df_chunks = spark.createDataFrame(all_chunk_rows, schema=chunks_schema)
    df_chunks.write.format("delta").mode("append").save(PATHS["chunks"])
    logger.info(f"Appended {len(all_chunk_rows)} chunks to Delta table.")

In [0]:
# Compress and archive successfully processed files, then remove them from the inbox
archive_count = 0
for file_path, file_type, file_hash, filename in successful_files:
    try:
        archive_dir = PATHS["archive_pdf"] if file_type == "PDF" else PATHS["archive_csv"]
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive_filename = f"{filename}.{ts}.gz"
        archive_path = os.path.join(archive_dir, archive_filename)
        
        with open(file_path, 'rb') as f_in:
            with gzip.open(archive_path, 'wb', compresslevel=9) as f_out:
                f_out.writelines(f_in)
                
        if os.path.exists(archive_path) and os.path.getsize(archive_path) > 0:
            os.remove(file_path)
            file_manifest[file_hash] = datetime.now(timezone.utc).isoformat()
            archive_count += 1
        else:
            logger.error(f"Archive verification failed for {archive_path}")
    except Exception as e:
        logger.error(f"Failed to archive {filename}: {e}")

save_manifest("processed_files", file_manifest)
logger.info(f"Archived {archive_count} files successfully.")

## Stage 2: LLM Question Generation

Generate synthetic training questions for each chunk using NVIDIA's LLM API. Implements batch processing with rate limiting and robust retry logic.

In [0]:
# Configure OpenAI-compatible client for NVIDIA API with automatic retry logic
openai_client = openai.OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=api
)

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError, openai.APITimeoutError, openai.APIConnectionError)),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    stop=stop_after_attempt(5)
)
def call_nvidia_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 1024) -> tuple:
    """Call NVIDIA LLM and return (content, finish_reason, usage_info)."""
    response = openai_client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.3
    )
    
    content = response.choices[0].message.content
    finish_reason = response.choices[0].finish_reason
    usage = response.usage if hasattr(response, 'usage') else None
    
    return content, finish_reason, usage


In [0]:
# Define prompt template for batch question generation
# Processes multiple chunks in a single API call to optimize throughput
BATCH_QUESTION_PROMPT_TEMPLATE = """
You are an expert curriculum designer. You will receive multiple text chunks, each labeled with a unique chunk_id.
For EACH chunk, generate exactly {n} questions that can be answered SOLELY using the information in that specific chunk.
Do not use outside knowledge.

Output strictly valid JSON and nothing else. No markdown fences, no commentary.
Format must be a JSON object mapping chunk_id to a list of question objects:
{{
  "chunk_id_1": [
    {{"question": "...", "answerable_from_chunk": true}},
    {{"question": "...", "answerable_from_chunk": true}}
  ],
  "chunk_id_2": [
    {{"question": "...", "answerable_from_chunk": true}}
  ]
}}

Chunks:
{chunks_data}
"""

In [0]:
# Core function to generate questions for multiple chunks in a single API call
# Includes robust JSON parsing, truncation handling, and automatic fallback to smaller batches
def attempt_parse_truncated_json(json_str: str, expected_chunks: int) -> dict:
    """
    Attempt to parse truncated JSON by finding complete chunk entries.
    Returns dict of successfully parsed chunk_id -> questions mappings.
    """
    results = {}
    
    # Try to extract complete chunk entries using pattern matching
    # Pattern: "chunk_id": [{...}, {...}, {...}]
    chunk_pattern = r'"([a-f0-9\-]+)"\s*:\s*\[((?:[^\[\]]*\{[^}]*\}[^\[\]]*,?\s*)+)\]'
    
    matches = re.finditer(chunk_pattern, json_str)
    for match in matches:
        chunk_id = match.group(1)
        questions_str = '[' + match.group(2) + ']'
        
        try:
            # Clean up trailing commas before closing bracket
            questions_str = re.sub(r',\s*\]', ']', questions_str)
            questions_list = json.loads(questions_str)
            
            if isinstance(questions_list, list) and len(questions_list) > 0:
                results[chunk_id] = questions_list
        except json.JSONDecodeError:
            continue
    
    return results if results else None

def generate_questions_for_batch(chunk_batch: list, n: int, retry_smaller: bool = True) -> dict:
    """
    Generate questions for a batch of chunks in a single API call.
    
    Args:
        chunk_batch: List of Row objects with chunk_id, doc_id, chunk_text
        n: Number of questions per chunk
    
    Returns:
        Dict mapping chunk_id -> list of question dicts
    """
    if not chunk_batch:
        return {}
    
    # Format chunks for the prompt
    chunks_data = ""
    for row in chunk_batch:
        chunks_data += f"\n--- CHUNK_ID: {row.chunk_id} ---\n{row.chunk_text}\n"
    
    prompt = BATCH_QUESTION_PROMPT_TEMPLATE.format(n=n, chunks_data=chunks_data)
    system = "You are a strict JSON data generator."
    
    # Calculate appropriate max_tokens based on actual input size
    # Estimate tokens: ~4 chars per token for prompt, output needs ~50 tokens per question
    prompt_tokens = len(prompt) // 4
    output_tokens = 100 * n * len(chunk_batch)  # 100 tokens per question (generous buffer)
    max_tokens = min(MAX_MODEL_TOKENS - prompt_tokens, output_tokens + 2000)
    
    try:
        raw_output, finish_reason, usage = call_nvidia_llm(prompt=prompt, system=system, max_tokens=max_tokens)
        
        # Check if response was truncated
        if finish_reason == 'length':
            logger.warning(f"Response truncated (hit max_tokens={max_tokens}) for batch of {len(chunk_batch)} chunks")
            # Will attempt to parse partial response and retry missing chunks later
        
        if usage:
            logger.debug(f"Token usage: prompt={usage.prompt_tokens}, completion={usage.completion_tokens}, total={usage.total_tokens}")
    except Exception as e:
        logger.error(f"LLM API call failed for batch of {len(chunk_batch)} chunks: {e}")
        return {}
    
    # Parse JSON response with robust extraction
    parsed = None
    
    # Try 1: Direct JSON parsing
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError:
        pass
    
    # Try 2: Remove markdown code fences if present
    if parsed is None:
        cleaned = raw_output.strip()
        if cleaned.startswith('```'):
            # Remove opening fence
            lines = cleaned.split('\n')
            if lines[0].startswith('```'):
                lines = lines[1:]
            # Remove closing fence
            if lines and lines[-1].strip() == '```':
                lines = lines[:-1]
            cleaned = '\n'.join(lines)
            
            try:
                parsed = json.loads(cleaned)
            except json.JSONDecodeError:
                pass
    
    # Try 3: Extract JSON object using regex (handles text before/after JSON)
    if parsed is None:
        # Match outermost curly braces with anything inside
        match = re.search(r'\{[\s\S]*\}', raw_output)
        if match:
            json_str = match.group(0)
            try:
                parsed = json.loads(json_str)
            except json.JSONDecodeError as e:
                # Try 4: Handle truncated JSON - attempt to salvage partial results
                if finish_reason == 'length':
                    logger.warning(f"Attempting to parse truncated JSON...")
                    parsed = attempt_parse_truncated_json(json_str, len(chunk_batch))
                    if parsed:
                        logger.info(f"Successfully parsed {len(parsed)} chunks from truncated response")
                else:
                    logger.error(f"Regex extraction found JSON-like text but parsing failed: {e}")
                    sample = json_str[:500] + '...' if len(json_str) > 500 else json_str
                    logger.error(f"Extracted text sample: {sample}")
        else:
            logger.error(f"No JSON object pattern found in output")
    
    # If all parsing attempts failed
    if parsed is None:
        logger.error(f"All JSON parsing attempts failed for batch of {len(chunk_batch)} chunks")
        # Log first 1000 chars of raw output for debugging
        sample = raw_output[:1000] + '...' if len(raw_output) > 1000 else raw_output
        logger.error(f"Raw output sample: {sample}")
        return {}
    
    # Validate structure: should be dict with chunk_ids as keys
    if not isinstance(parsed, dict):
        logger.error(f"Expected dict, got {type(parsed)}")
        return {}
    
    # Validate and clean the results
    validated_results = {}
    for chunk_id, questions in parsed.items():
        if not isinstance(questions, list):
            logger.warning(f"Chunk {chunk_id}: expected list of questions, got {type(questions)}")
            continue
        
        valid_questions = []
        for item in questions:
            if isinstance(item, dict):
                q = item.get("question", "").strip()
                if q and item.get("answerable_from_chunk") is True:
                    valid_questions.append(item)
        
        if valid_questions:
            validated_results[chunk_id] = valid_questions
    
    # Check if we got results for all chunks in batch
    successful_chunks = len(validated_results)
    total_chunks = len(chunk_batch)
    
    if successful_chunks < total_chunks:
        missing_count = total_chunks - successful_chunks
        logger.warning(f"Batch incomplete: got results for {successful_chunks}/{total_chunks} chunks ({missing_count} missing)")
        
        # If we got partial results and retry is enabled, try smaller batches for missing chunks
        if retry_smaller and missing_count > 0 and missing_count < total_chunks:
            processed_chunk_ids = set(validated_results.keys())
            missing_chunks = [row for row in chunk_batch if row.chunk_id not in processed_chunk_ids]
            
            logger.info(f"Retrying {missing_count} missing chunks in smaller batches of {FALLBACK_BATCH_SIZE}...")
            
            # Split missing chunks into FALLBACK_BATCH_SIZE batches
            for i in range(0, len(missing_chunks), FALLBACK_BATCH_SIZE):
                retry_batch = missing_chunks[i:i+FALLBACK_BATCH_SIZE]
                logger.info(f"  Retry sub-batch {i//FALLBACK_BATCH_SIZE + 1}: {len(retry_batch)} chunks")
                
                try:
                    # Recursive call with retry_smaller=False to avoid infinite recursion
                    retry_results = generate_questions_for_batch(retry_batch, n, retry_smaller=False)
                    validated_results.update(retry_results)
                    logger.info(f"    ✓ Recovered {len(retry_results)} chunks")
                except Exception as e:
                    logger.error(f"    ✗ Retry sub-batch failed: {e}")
    
    return validated_results

In [0]:
# Execute parallel question generation with wave-based rate limiting
# Processes chunks in batches, respecting API rate limits and writing results incrementally
import time
from queue import Queue

questions_schema = StructType([
    StructField("question_id", StringType(), False),
    StructField("chunk_id", StringType(), False),
    StructField("doc_id", StringType(), False),
    StructField("question_text", StringType(), False),
    StructField("model_used", StringType(), False),
    StructField("created_at_utc", TimestampType(), False)
])

# Initialize variables (prevents NameError in Cell 17)
chunks_df = None
pending_chunks = []
all_questions = []  # Accumulates ALL questions across all waves
failed_questions = 0
failed_batches = 0
total_api_calls = 0

try:
    chunks_df = spark.read.format("delta").load(PATHS["chunks"])
    logger.info("Successfully loaded chunks table")
except Exception as e:
    logger.warning(f"Chunks table not found or empty: {e}")

if chunks_df:
    # Load manifest of already-processed chunks
    chunks_manifest = load_manifest("processed_chunks_for_questions")
    
    # Get un-processed chunks
    all_chunks = chunks_df.select("chunk_id", "doc_id", "chunk_text")
    total_chunks = all_chunks.count()
    logger.info(f"Total chunks in table: {total_chunks}")
    logger.info(f"Already processed: {len(chunks_manifest)}")
    
    # Collect only unprocessed chunks
    pending_chunks = [
        row for row in all_chunks.collect()
        if row.chunk_id not in chunks_manifest
    ]
    
    num_pending = len(pending_chunks)
    logger.info(f"Pending chunks: {num_pending}")
    
    if num_pending == 0:
        logger.info("No pending chunks to process. All done!")
    else:
        # Calculate batching metrics
        num_batches = (num_pending + BATCH_SIZE - 1) // BATCH_SIZE
        num_waves = (num_batches + API_RATE_LIMIT - 1) // API_RATE_LIMIT
        estimated_time_min = num_waves  # ~1 minute per wave
        
        logger.info("="*70)
        logger.info(f"PARALLEL BATCH PROCESSING CONFIGURATION")
        logger.info(f"  Chunks per batch: {BATCH_SIZE}")
        logger.info(f"  Total batches: {num_batches}")
        logger.info(f"  Parallel calls per wave: {API_RATE_LIMIT}")
        logger.info(f"  Total waves: {num_waves}")
        logger.info(f"  Estimated time: ~{estimated_time_min} minutes")
        logger.info("="*70)
        
        # Process all batches in waves of API_RATE_LIMIT parallel calls
        start_time = time.time()
        
        def process_single_batch(batch_idx, chunk_batch):
            """Process one batch - called in parallel threads."""
            batch_num = batch_idx + 1
            call_start = time.time()
            
            try:
                # Make API call
                batch_results = generate_questions_for_batch(chunk_batch, QUESTIONS_PER_CHUNK)
                call_duration = time.time() - call_start
                
                # Process results and create question records
                batch_questions = []
                now_utc = datetime.now(timezone.utc)
                failed_in_batch = 0
                
                for row in chunk_batch:
                    chunk_id = row.chunk_id
                    doc_id = row.doc_id
                    
                    if chunk_id in batch_results:
                        questions_list = batch_results[chunk_id]
                        for i, q_dict in enumerate(questions_list):
                            q_id = str(uuid.uuid5(NAMESPACE, f"{chunk_id}:q{i}"))
                            batch_questions.append({
                                "question_id": q_id,
                                "chunk_id": chunk_id,
                                "doc_id": doc_id,
                                "question_text": q_dict["question"],
                                "model_used": NVIDIA_MODEL,
                                "created_at_utc": now_utc
                            })
                    else:
                        failed_in_batch += 1
                
                return {
                    "success": True,
                    "batch_num": batch_num,
                    "questions": batch_questions,
                    "chunk_ids": [row.chunk_id for row in chunk_batch],
                    "failed_chunks": failed_in_batch,
                    "duration": call_duration,
                    "timestamp": now_utc
                }
                
            except Exception as e:
                logger.error(f"Batch {batch_num} failed: {e}")
                return {
                    "success": False,
                    "batch_num": batch_num,
                    "error": str(e),
                    "chunk_ids": [row.chunk_id for row in chunk_batch]
                }
        
        # Process in waves
        for wave_idx in range(num_waves):
            wave_num = wave_idx + 1
            wave_start_time = time.time()
            
            # Determine batches for this wave
            wave_start_batch = wave_idx * API_RATE_LIMIT
            wave_end_batch = min(wave_start_batch + API_RATE_LIMIT, num_batches)
            batches_in_wave = wave_end_batch - wave_start_batch
            
            logger.info(f"\n{'='*70}")
            logger.info(f"WAVE {wave_num}/{num_waves} - Processing {batches_in_wave} batches in parallel")
            logger.info(f"{'='*70}")
            
            # Prepare batch data for this wave
            wave_futures = []
            with ThreadPoolExecutor(max_workers=API_RATE_LIMIT) as executor:
                for batch_idx in range(wave_start_batch, wave_end_batch):
                    batch_start_idx = batch_idx * BATCH_SIZE
                    batch_end_idx = min(batch_start_idx + BATCH_SIZE, num_pending)
                    chunk_batch = pending_chunks[batch_start_idx:batch_end_idx]
                    
                    # Submit batch for parallel processing
                    future = executor.submit(process_single_batch, batch_idx, chunk_batch)
                    wave_futures.append(future)
                
                # Process results as they complete (dynamic processing during wait)
                wave_questions = []  # Questions collected in this wave
                completed = 0
                
                logger.info(f"Submitted {len(wave_futures)} parallel API calls...")
                logger.info("Processing responses as they arrive...")
                
                for future in as_completed(wave_futures):
                    completed += 1
                    result = future.result()
                    total_api_calls += 1
                    
                    if result["success"]:
                        # Add questions to wave collection
                        wave_questions.extend(result["questions"])
                        
                        # Update manifest for processed chunks
                        for chunk_id in result["chunk_ids"]:
                            chunks_manifest[chunk_id] = result["timestamp"].isoformat()
                        
                        failed_questions += result["failed_chunks"]
                        
                        logger.info(
                            f"  ✓ Batch {result['batch_num']}: "
                            f"{len(result['questions'])} questions "
                            f"({result['duration']:.2f}s) "
                            f"[{completed}/{batches_in_wave} complete]"
                        )
                    else:
                        failed_batches += 1
                        logger.error(f"  ✗ Batch {result['batch_num']} failed: {result['error']}")
                
                # Wave complete - all responses received
                wave_duration = time.time() - wave_start_time
                logger.info(f"\nWave {wave_num} completed in {wave_duration:.2f}s")
                logger.info(f"  Questions generated: {len(wave_questions)}")
                logger.info(f"  Successful batches: {completed - failed_batches}")
                
                # Append wave questions to master list
                all_questions.extend(wave_questions)
                
                # Create DataFrame from wave questions and save
                if wave_questions:
                    df_wave = spark.createDataFrame(wave_questions, schema=questions_schema)
                    df_wave.write.format("delta").mode("append").save(PATHS["questions"])
                    save_manifest("processed_chunks_for_questions", chunks_manifest)
                    
                    logger.info(f"  ✓ Saved {len(wave_questions)} questions to Delta table")
                
                # Progress update
                elapsed_min = (time.time() - start_time) / 60.0
                progress_pct = (wave_num / num_waves) * 100
                total_qs_so_far = len(all_questions)
                logger.info(f"\nOverall Progress: {progress_pct:.1f}% | Elapsed: {elapsed_min:.2f}min | Total Questions: {total_qs_so_far}")
            
            # Rate limiting between waves (if not last wave)
            if wave_num < num_waves:
                # Calculate time to wait before next wave
                time_elapsed_in_wave = time.time() - wave_start_time
                wait_time = max(0, MIN_WAVE_INTERVAL - time_elapsed_in_wave)
                
                if wait_time > 0:
                    logger.info(f"\n⏱️  Rate limit: waiting {wait_time:.1f}s before next wave...")
                    time.sleep(wait_time)
                else:
                    logger.info(f"\n⚡ Wave took >{MIN_WAVE_INTERVAL}s, proceeding immediately to next wave")
        
        # Final summary
        total_time_min = (time.time() - start_time) / 60.0
        logger.info("\n" + "="*70)
        logger.info(f"PARALLEL BATCH PROCESSING COMPLETE")
        logger.info(f"  Total time: {total_time_min:.2f} minutes")
        logger.info(f"  Total API calls: {total_api_calls}")
        logger.info(f"  Successful batches: {num_batches - failed_batches}/{num_batches}")
        logger.info(f"  Failed batches: {failed_batches}")
        logger.info(f"  Total questions generated: {len(all_questions)}")
        logger.info(f"  Chunks processed: {len(chunks_manifest)}")
        logger.info(f"  Individual chunk failures: {failed_questions}")
        logger.info(f"  Throughput: {total_api_calls / total_time_min:.1f} calls/min")
        logger.info("="*70)

## Validation & Summary

Validate output tables and display processing statistics.

In [0]:
# Validate chunks and questions tables: check counts, null values, and display samples
try:
    df_c = spark.read.format("delta").load(PATHS["chunks"])
    df_q = spark.read.format("delta").load(PATHS["questions"])
    
    logger.info(f"Total Chunks: {df_c.count()}")
    logger.info(f"Total Questions: {df_q.count()}")
    
    # Null checks
    null_chunks = df_c.filter(F.col("chunk_text").isNull()).count()
    null_qs = df_q.filter(F.col("question_text").isNull()).count()
    assert null_chunks == 0, "Found nulls in chunk_text"
    assert null_qs == 0, "Found nulls in question_text"
    
    print("Sample Chunks:")
    display(df_c.limit(5))
    
    print("Sample Questions:")
    display(df_q.limit(5))
except Exception as e:
    logger.error(f"Validation failed (tables might be empty): {e}")

In [0]:
# Display comprehensive summary of both processing stages with metrics
print("="*60)
print("RUN SUMMARY: NOTEBOOK 1 - CHUNKING & QUESTION GENERATION")
print("="*60)

# Stage 1: File Ingestion & Chunking
print("\n[STAGE 1: FILE INGESTION & CHUNKING]")
try:
    print(f"  Files Discovered: {len(new_files)}")
    print(f"  Files Chunked Successfully: {len(successful_files)}")
    print(f"  Files Archived: {archive_count}")
    print(f"  Files Failed: {len(failed_files)}")
except NameError:
    print("  Stage 1 not executed (run cells 7-11)")

# Stage 2: Question Generation
print("\n[STAGE 2: QUESTION GENERATION]")
try:
    if chunks_df is not None:
        print(f"  Chunks Available: Yes")
        print(f"  Pending Chunks Processed: {len(pending_chunks)}")
        print(f"  Questions Generated: (saved incrementally to Delta)")
        print(f"  Individual Chunk Failures: {failed_questions}")
        if 'failed_batches' in dir():
            print(f"  Batch Failures: {failed_batches}")
    else:
        print("  Chunks Available: No (chunks table not found)")
except NameError:
    print("  Stage 2 not executed (run cell 15)")

print("\n" + "="*60)
print("To validate outputs, run Cell 16 (Validation)")
print("="*60)